# Gas-Water (Air-Water) Generalization Check -- Sister Notebook

Split out from `mixtureTheoryFlow_jax.ipynb` (2026-10-28) to keep the two
fluid-pair stories separate: that notebook develops and calibrates the
closure against oil-water data (Ibarra, Russell, Angeli); this one applies
the resulting, fully Ibarra-fitted closure *zero-refit* against a
categorically different fluid pair -- air/water -- per CLAUDE.md's
"Phase 2g: Gas-Water Generalization Check".

Two reasons for the split, not just preference: (1) the main notebook
already hit a size crisis once (568KB, forcing the core solver out into
`mixtureTheoryFlow_jax_solver.py`), and piling a second fluid pair's worth
of sections onto it risked repeating that; (2) since the solver is now a
standalone module, this notebook costs nothing in duplication -- it
imports the exact same code the main notebook does.

**Status**: both gas-water datasets here (MDPI/Carraretto, Fan 2005) turn
out to be dominated by `PHI_COLLAPSE`-triggered conditions -- see
CLAUDE.md's "Current Limitation: Vanishing-Phase Singularity Dominates
(Not Just Bounds) Gas-Liquid Data". Work is paused pending that decision
(lower `PHI_COLLAPSE`, find higher-holdup gas-liquid data, or revisit the
vanishing-phase formulation itself) -- the forward Um/psi/dP-dL validation
and any recalibration (steps 4 and 6 of the Phase 2g plan) are blocked
until it's resolved.


In [ ]:
# Same core solver as mixtureTheoryFlow_jax.ipynb -- no duplication, this
# notebook is a second consumer of the one shared module, not a fork of it.
from mixtureTheoryFlow_jax_solver import *


## Shared helper functions (quiet-window classification)

Duplicated from the main notebook's `p2a-quiet-window-01` cell, unchanged
-- this notebook has its own kernel/namespace, so `p2g-quiet-window-01` and
`p2g-fan2005-osc-01` below need their own copy of
`find_quiet_streaks`/`compute_developed_mask` rather than relying on a
cell that ran earlier in a *different* notebook's session.


In [ ]:
def find_quiet_streaks(quiet_mask):
    """Return list of (start_idx, end_idx_inclusive, length) for contiguous True runs."""
    streaks = []
    start = None
    for i, q in enumerate(quiet_mask):
        if q and start is None:
            start = i
        elif not q and start is not None:
            streaks.append((start, i - 1, i - start))
            start = None
    if start is not None:
        streaks.append((start, len(quiet_mask) - 1, len(quiet_mask) - start))
    return streaks


def compute_developed_mask(Um_sim_c, Um_target, valid, lookback, rel_tol, min_fraction):
    """A chunk is developed if Um_sim clears a loose floor (flow has actually
    started) AND has stopped changing over the trailing `lookback` chunks
    (flow has actually finished converging, not just started ramping up)."""
    n = len(Um_sim_c)
    developed = np.zeros(n, dtype=bool)
    for i in range(n):
        if i < lookback - 1 or not valid[i]:
            continue   # not enough trailing history yet, or this chunk itself is invalid
        window       = Um_sim_c[i - lookback + 1 : i + 1]
        window_valid = valid[i - lookback + 1 : i + 1]
        if not np.all(window_valid):
            continue
        started_flowing = Um_sim_c[i] >= min_fraction * Um_target
        um_range         = np.max(window) - np.min(window)
        settled          = um_range < rel_tol * Um_target
        developed[i]     = started_flowing and settled
    return developed


## Reset globals to Carraretto et al. (2020) MDPI air-water parameters

In [ ]:
# ── Reset globals to Carraretto et al. (2020) MDPI parameters (air / water, PMMA pipe) ──
L          = 24.0       # pipe length [m] -- full 24m PMMA test section
D          = 0.060      # pipe diameter [m] -- 60mm ID
N          = 500        # number of cells
theta      = 0.0        # horizontal
rho1_val   = 998.0      # water [kg/m^3]
rho2_val   = 1.204      # air [kg/m^3] -- 20C, 1 atm
mu1        = 1.0E-3     # water [Pa.s] -- 20C
mu2        = 1.81E-5    # air [Pa.s] -- 20C, 1 atm
drag_coeff = 1E-04
d_b        = 1e-3

print(f"Globals set to MDPI (Carraretto et al. 2020) air-water:  L={L}  D={D}  N={N}  "
      f"rho1_val={rho1_val}  rho2_val={rho2_val}  mu1={mu1}  mu2={mu2}")


# Phase 2g: Gas-Water Generalization Check (Carraretto et al. 2020, MDPI)

Zero-refit stress test of the fully Ibarra-fitted closure (Taitel-Dukler
geometry + S_i/w + k1/k2 + C_D) against a categorically different fluid
pair -- air/water, not another oil/water dataset. Data: Carraretto et al.
(2020), *Pressure Drop and Void Fraction in Horizontal Air-Water Stratified
Flows with Smooth Interface at Atmospheric Pressure*, 24m/60mm PMMA pipe,
digitized from Figures 6, 9, 10 (`MDPI_JL_03/04/05/06.csv`, combined into
`mdpi_airwater_phase2g_dataset.csv`).

Interposed before continuing Phase 2c (see CLAUDE.md "New intermediate
phase decided" note). Scoped horizontal-only, stratified-smooth-only (per
the paper's own operating range) -- no additional flow-regime filtering
needed here, unlike the planned Beggs & Brill alternative.

**Known risk going in**: nearly every condition has WC < 0.1 (water is a
small minority phase at these liquid loadings), right at/below
PHI_COLLAPSE -- the oscillation/collapse sweep below runs BEFORE any
forward Um/psi validation, to see how many conditions are even usable
under the current two-fluid formulation.

## Load the combined MDPI Phase 2g dataset (Figures 6/9/10, digitized)

In [ ]:
# ── Load Carraretto et al. (2020) MDPI air-water stratified flow data ──
# Built from MDPI_JL_03/04/05/06.csv (digitized Figures 6, 9, 10) --
# combined per-condition (Um_target, WC, dpdz_measured, psi) already in the
# same schema Ibarra/Russell use. flow_regime flags the 4 highest-JG
# conditions (one per JL series, JG~2.35-2.4 m/s) as near the paper's own
# ~2 m/s 2D-waviness onset -- informational only, not excluded here.

import pandas as pd

mdpi_df = pd.read_csv('mdpi_airwater_phase2g_dataset.csv')
mdpi_clean = mdpi_df[['Um_target', 'WC', 'dpdz_measured', 'psi', 'flow_regime']].copy()

print(f"Loaded {len(mdpi_clean)} conditions from Carraretto et al. (2020)")
print(f"WC range: {mdpi_clean['WC'].min():.4f}-{mdpi_clean['WC'].max():.4f}  "
      f"(PHI_COLLAPSE={PHI_COLLAPSE} -- most conditions sit at or below this)")
print(mdpi_clean.to_string())


## Diagnostic: sweep ALL MDPI conditions for oscillation + collapse (zero-refit Ibarra-fitted closure)

In [ ]:
# ── Diagnostic: sweep ALL MDPI conditions for oscillation + collapse ──
# Same methodology as the Ibarra/Russell sweeps, applied to MDPI. Zero-refit:
# uses the Ibarra-fitted w/k1/k2 closure unchanged, per the Phase 2g plan.
# Tracks per chunk:
#   phi2_range -- oscillation proxy (spike-and-crash near WC=0.7 on Ibarra).
#   Um_sim     -- distinguishes "never started flowing" from "settled".
#   phi_min    -- PHI_COLLAPSE check -- expected to trigger often here, given
#                 nearly every MDPI condition has WC < 0.1 (see Known Risk above).
# Domain-wide scalars only, to keep memory bounded.
#
# Nondimensional boundary conversion (same as generate_real_dataset): D*=1.0
# (L_c=D by construction), p*=p/(rho1*Um^2), d_b*=d_b/D, dt*=dt_seconds*Um/D,
# Re1_ref/Re2_ref/Fr_ref computed per condition from that condition's own
# Um_target -- time_step_learned's viscous-stress term uses 1/Re_ref
# internally now, not mu1/mu2 directly, so these are REQUIRED, not optional
# (their None default raises TypeError -- see CLAUDE.md's nondimensionalization note).
#
# REMINDER: D/L/rho1_val/rho2_val/mu1/mu2 must already be set to MDPI's
# values -- see reset-globals-mdpi-01 above.

import numpy as np
import pickle

MDPI_W, MDPI_K1, MDPI_K2 = 1.0, 1.1102, 0.5853   # Ibarra-fitted, zero-refit

N_TOTAL_STEPS_SWEEP       = 370_000   # ~37s -- same full-duration convention as Russell
n_chunks_sweep            = 50
chunk_steps_sweep         = N_TOTAL_STEPS_SWEEP // n_chunks_sweep
phi2_range_flag_threshold = 0.3   # coarse placeholder -- see Ibarra/Russell sweeps

# Dataset-level dimensionless quantities (don't depend on Um, computed once)
D_star    = 1.0
L_star    = L / D
rho1_star = 1.0
rho2_star = rho2_val / rho1_val
d_b_star  = d_b / D

oscillation_sweep_results_mdpi = []

for _, row in mdpi_clean.iterrows():
    WC_sw, dpdz_sw, Um_sw, psi_sw = (
        row['WC'], row['dpdz_measured'], row['Um_target'], row['psi']
    )

    print(f"\n{'='*76}")
    print(f"Condition: WC={WC_sw:.4f}  Um={Um_sw:.2f}  dpdz={dpdz_sw:.2f}  psi={psi_sw:.4f}")
    print(f"{'='*76}")

    p_out_sw = 1.0000e5
    p_in_sw  = p_out_sw + dpdz_sw * L

    # Per-condition reference scales -- Um_sw is this condition's own
    # experimentally-known mixture velocity.
    P_c        = rho1_val * Um_sw**2
    p_out_star = p_out_sw / P_c
    p_in_star  = p_in_sw  / P_c
    Re1_ref    = rho1_val * Um_sw * D / mu1
    Re2_ref    = rho1_val * Um_sw * D / mu2
    Fr_ref     = Um_sw / jnp.sqrt(9.81 * D)
    dt_star    = dt_fixed * Um_sw / D

    dx_sw, x_sw = make_grid(L_star, N)
    state_sw = initial_conditions(x_sw, L_star, WC_sw, rho1_star, rho2_star, p_in_star, p_out_star)

    def sweep_scan_step(state, dt):
        new_state = time_step_learned(
            state, dt, dx_sw, drag_coeff, D_star,
            p_in_star, p_out_star, d_b_star, mu1, mu2, theta, WC_sw,
            w=MDPI_W, k1=MDPI_K1, k2=MDPI_K2,
            Re1_ref=Re1_ref, Re2_ref=Re2_ref, Fr_ref=Fr_ref
        )
        return new_state, None

    sweep_scan_jit = jax.jit(lambda s: jax.lax.scan(
        sweep_scan_step, s, jnp.full(chunk_steps_sweep, dt_star)
    )[0])

    t_arr          = np.full(n_chunks_sweep, np.nan)
    phi2_range_arr = np.full(n_chunks_sweep, np.nan)
    slip_range_arr = np.full(n_chunks_sweep, np.nan)
    Um_sim_arr     = np.full(n_chunks_sweep, np.nan)
    phi_min_arr    = np.full(n_chunks_sweep, np.nan)

    i_start_sw, i_end_sw = N // 4, 3 * N // 4

    state_sw_run  = state_sw
    hit_nan_chunk = None

    for c in range(n_chunks_sweep):
        state_sw_run = sweep_scan_jit(state_sw_run)
        state_sw_run['phi1'].block_until_ready()

        has_nan = any(bool(jnp.any(jnp.isnan(v))) for v in state_sw_run.values())
        if has_nan:
            hit_nan_chunk = c
            print(f"  NaN encountered at chunk {c} -- stopping this condition's sweep")
            break

        phi1_c = state_sw_run['phi1']
        phi2_c = state_sw_run['phi2']
        u1_c   = state_sw_run['u1']
        u2_c   = state_sw_run['u2']
        slip_c = u2_c - u1_c

        t_arr[c]          = (c + 1) * chunk_steps_sweep * dt_fixed
        phi2_range_arr[c] = float(jnp.max(phi2_c) - jnp.min(phi2_c))
        slip_range_arr[c] = float(jnp.max(slip_c) - jnp.min(slip_c))
        Um_sim_arr[c]     = float(jnp.mean(
            phi1_c[i_start_sw:i_end_sw] * u1_c[i_start_sw:i_end_sw]
            + phi2_c[i_start_sw:i_end_sw] * u2_c[i_start_sw:i_end_sw]
        ))
        phi_min_arr[c]    = float(jnp.minimum(jnp.min(phi1_c), jnp.min(phi2_c)))

    if np.all(np.isnan(phi2_range_arr)):
        peak_phi2_range = float('nan')
        t_peak          = float('nan')
        oscillating     = False
    else:
        peak_phi2_range = float(np.nanmax(phi2_range_arr))
        t_peak          = float(t_arr[int(np.nanargmax(phi2_range_arr))])
        oscillating     = peak_phi2_range > phi2_range_flag_threshold

    final_Um_sim = float(Um_sim_arr[~np.isnan(Um_sim_arr)][-1]) if np.any(~np.isnan(Um_sim_arr)) else float('nan')

    print(f"  peak phi2_range = {peak_phi2_range:.4f} at t={t_peak:.1f}s   "
          f"flag(>{phi2_range_flag_threshold}) = {oscillating}   "
          f"Um_sim@end = {final_Um_sim:.4f} (target {Um_sw:.2f})"
          + (f"   [NaN at chunk {hit_nan_chunk}]" if hit_nan_chunk is not None else ""))

    oscillation_sweep_results_mdpi.append({
        'WC': WC_sw, 'Um': Um_sw, 'psi': psi_sw,
        't': t_arr, 'phi2_range': phi2_range_arr, 'slip_range': slip_range_arr,
        'Um_sim': Um_sim_arr, 'phi_min': phi_min_arr,
        'peak_phi2_range': peak_phi2_range, 't_peak': t_peak,
        'oscillating_flag': oscillating, 'nan_chunk': hit_nan_chunk,
    })

# ── Summary table ────────────────────────────────────────────────────────────
print(f"\n{'='*76}")
print("Summary across all MDPI conditions")
print(f"{'='*76}")
print(f"{'WC':>7}  {'Um':>5}  {'peak phi2_range':>16}  {'t_peak':>7}  {'flag':>6}  {'nan_chunk':>9}")
for r in oscillation_sweep_results_mdpi:
    print(f"{r['WC']:>7.4f}  {r['Um']:>5.2f}  {r['peak_phi2_range']:>16.4f}  "
          f"{r['t_peak']:>7.1f}  {str(r['oscillating_flag']):>6}  "
          f"{str(r['nan_chunk']) if r['nan_chunk'] is not None else '-':>9}")

# ── Save raw per-chunk time series for reuse by the classification step below ──
with open('mdpi_oscillation_sweep_results.pkl', 'wb') as f:
    pickle.dump(oscillation_sweep_results_mdpi, f)
print("\nSaved per-chunk time series to mdpi_oscillation_sweep_results.pkl")


## Locate quiet windows per MDPI condition, from the saved oscillation sweep

In [ ]:
# ── Diagnostic: locate a quiet, DEVELOPED training window per MDPI condition ──
# Same methodology/thresholds as the Ibarra/Russell quiet-window
# classification -- reuses find_quiet_streaks and compute_developed_mask
# (defined earlier in the notebook, unchanged). Chunk duration matches
# Russell's (N_TOTAL_STEPS_SWEEP=370,000 / 50 chunks = ~0.74s/chunk), so
# Russell's thresholds are reused as-is (already scaled for this duration).

QUIET_THRESHOLD        = 0.15   # phi2_range below this = spatially quiet chunk
DEVELOPED_MIN_FRACTION = 0.05   # floor: rules out "flow never started"
DEVELOPED_LOOKBACK     = 12     # trailing chunks for Um_sim stability (~9s at 0.74s/chunk)
DEVELOPED_REL_TOL      = 0.05   # Um_sim's range over the lookback window must be below this fraction of Um_target
MIN_QUIET_CHUNKS       = 8      # minimum contiguous usable chunks (~6s at 0.74s/chunk)
MIN_QUIET_FRACTION     = 0.15   # minimum share of the whole run that must be quiet AND developed
# PHI_COLLAPSE reused globally, same threshold the simulation itself uses.

mdpi_quiet_window_selection = []

for r in oscillation_sweep_results_mdpi:
    phi2_range_c = r['phi2_range']
    Um_sim_c     = r['Um_sim']
    phi_min_c    = r['phi_min']
    t_c          = r['t']
    Um_target    = r['Um']
    valid        = ~np.isnan(phi2_range_c)

    quiet_mask     = np.where(valid, phi2_range_c < QUIET_THRESHOLD, False)
    developed_mask = compute_developed_mask(
        Um_sim_c, Um_target, valid, DEVELOPED_LOOKBACK, DEVELOPED_REL_TOL, DEVELOPED_MIN_FRACTION
    )
    collapsed_mask = np.where(valid, phi_min_c < PHI_COLLAPSE, False)
    usable_mask    = quiet_mask & developed_mask & ~collapsed_mask

    streaks    = find_quiet_streaks(usable_mask)
    adequate   = [s for s in streaks if s[2] >= MIN_QUIET_CHUNKS]

    n_valid            = float(np.sum(valid)) if np.sum(valid) > 0 else 1.0
    quiet_fraction      = float(np.sum(quiet_mask))     / n_valid
    developed_fraction  = float(np.sum(developed_mask)) / n_valid
    collapsed_fraction  = float(np.sum(collapsed_mask)) / n_valid
    usable_fraction     = float(np.sum(usable_mask))    / n_valid

    if adequate and usable_fraction >= MIN_QUIET_FRACTION:
        start_idx, end_idx, length = adequate[-1]   # latest qualifying streak
        window_start = t_c[start_idx]
        window_end   = t_c[end_idx]
        excluded     = False
    else:
        window_start = window_end = None
        excluded     = True

    mdpi_quiet_window_selection.append({
        'WC': r['WC'], 'Um': r['Um'],
        'quiet_fraction': quiet_fraction, 'developed_fraction': developed_fraction,
        'collapsed_fraction': collapsed_fraction, 'usable_fraction': usable_fraction,
        'n_streaks': len(streaks), 'n_adequate_streaks': len(adequate),
        'window_start': window_start, 'window_end': window_end,
        'excluded': excluded,
    })

# ── Summary table ────────────────────────────────────────────────────────────
print(f"{'WC':>7}  {'Um':>5}  {'quiet':>7}  {'developed':>9}  {'collapsed':>9}  "
      f"{'usable':>7}  {'n_adeq':>6}  {'window':>16}")
for r in mdpi_quiet_window_selection:
    if r['excluded']:
        window_str = 'EXCLUDED'
    else:
        window_str = f"{r['window_start']:.1f}-{r['window_end']:.1f}s"
    print(f"{r['WC']:>7.4f}  {r['Um']:>5.2f}  {r['quiet_fraction']:>6.1%}  "
          f"{r['developed_fraction']:>8.1%}  {r['collapsed_fraction']:>8.1%}  "
          f"{r['usable_fraction']:>6.1%}  {r['n_adequate_streaks']:>6}  {window_str:>16}")

n_excluded = sum(r['excluded'] for r in mdpi_quiet_window_selection)
print(f"\n{n_excluded}/{len(mdpi_quiet_window_selection)} conditions excluded (no adequate quiet+developed+uncollapsed window).")

with open('mdpi_quiet_window_selection.pkl', 'wb') as f:
    pickle.dump(mdpi_quiet_window_selection, f)
print("Saved per-condition quiet window selection to mdpi_quiet_window_selection.pkl")


## Resets globals to Yan, F. 2005 SW flow dataset from KFUPM

In [5]:
# ── Reset globals to Fan, Y. (2005) parameters (air / water, 2in pipe) ──
# rho2_val (air density) is NOT set here -- unlike every prior dataset, it
# varies per condition (2.18-2.67+ kg/m^3 across the subset, depends on
# local pressure) and is carried as fan2005_clean's own rho2_val column
# instead, read per-row by whatever loop consumes it.
L          = 19.0       # pipe length [m] -- from the source database (Pipe Length[m])
D          = 0.0508     # pipe diameter [m] -- 2-inch pipe
N          = 500        # number of cells
theta      = 0.0        # horizontal subset only (Inclination angle == 0)
rho1_val   = 1000.0     # water [kg/m^3]
mu1        = 1.0E-3     # water [Pa.s]
mu2        = 1.8E-5     # air [Pa.s]
drag_coeff = 1E-04
d_b        = 1e-3

print(f"Globals set to Fan et al. (2005) air-water:  L={L}  D={D}  N={N}  "
      f"rho1_val={rho1_val}  mu1={mu1}  mu2={mu2}  "
      f"(rho2_val varies per-row -- see fan2005_clean)")

Globals set to Fan et al. (2005) air-water:  L=19.0  D=0.0508  N=500  rho1_val=1000.0  mu1=0.001  mu2=1.8e-05  (rho2_val varies per-row -- see fan2005_clean)


## Fan, Y. (2005) -- Low Liquid Loading Stratified-Wavy Air-Water (horizontal subset)

Second Phase 2g gas-water dataset, zero-refit against the same Ibarra-fitted
closure used for MDPI above. Source: `fan2005_sw_subset.csv`, filtered from
the FFPU entrainment database to Authors=="Fan, Y." and Flow Pattern=="SW"
(60 rows total). Starting with the 13 exactly-horizontal rows
(Inclination angle == 0); the other 47 span -2 deg to +2 deg, deferred for
now per the Phase 2g horizontal-only scoping decision.

**Expect severe collapse**: WC works out to ~3e-5 to 1.6e-3 here (vSL =
0.0007-0.024 m/s against vSG = 15-26 m/s) -- three to four orders of
magnitude below PHI_COLLAPSE=0.1, far more extreme than MDPI's 0.01-0.12
range. The oscillation/collapse sweep below is run first, same reasoning
as MDPI, before any forward Um/psi validation.

### Load Fan, Y. (2005) stratified-wavy (SW) subset

In [3]:
# ── Load Fan, Y. (2005) stratified-wavy (SW) subset ──
# Horizontal rows only (Inclination angle == 0) for this first pass.
# WC/Um_target are no-slip quantities built from vSL/vSG (same convention as
# every other dataset here); psi is HL directly (liquid holdup, same role)
# -- rows with no HL measurement ('-' in the raw data) get psi=NaN rather
# than being coerced to a number, and are excluded from holdup checks.
#
# rhoG is NOT constant for this dataset (depends on local pressure, unlike
# every prior dataset's single fixed rho2_val) -- kept as its own per-row
# column (rho2_val) and read per-condition downstream, not set as a
# notebook global the way rho1_val/mu1/mu2 are in reset-globals-fan2005-01.

import pandas as pd

fan2005_df = pd.read_csv('fan2005_sw_subset.csv')
fan2005_horiz = fan2005_df[fan2005_df['Inclination angle[°]'] == 0].copy()

fan2005_horiz['WC']            = fan2005_horiz['vSL[m/s]'] / (fan2005_horiz['vSL[m/s]'] + fan2005_horiz['vSG[m/s]'])
fan2005_horiz['Um_target']     = fan2005_horiz['vSL[m/s]'] + fan2005_horiz['vSG[m/s]']
fan2005_horiz['dpdz_measured'] = fan2005_horiz['dP/dL[Pa/m]']
fan2005_horiz['psi']           = pd.to_numeric(fan2005_horiz['HL[-]'], errors='coerce')  # '-' -> NaN
fan2005_horiz['rho2_val']      = fan2005_horiz['rhoG[kg/m3]']
fan2005_horiz['flow_regime']   = 'SW'

fan2005_clean = fan2005_horiz[[
    'Um_target', 'WC', 'dpdz_measured', 'psi', 'flow_regime', 'rho2_val',
]].reset_index(drop=True)

n_no_psi = fan2005_clean['psi'].isna().sum()
print(f"Loaded {len(fan2005_clean)} horizontal SW conditions from Fan, Y. (2005)")
print(f"  {n_no_psi}/{len(fan2005_clean)} have no holdup (HL) measurement -- excluded from holdup checks, Um-only")
print(f"  WC range: {fan2005_clean['WC'].min():.2e} - {fan2005_clean['WC'].max():.2e}  "
      f"(PHI_COLLAPSE={PHI_COLLAPSE} -- every condition sits far below this)")
print(fan2005_clean.to_string())


Loaded 13 horizontal SW conditions from Fan, Y. (2005)
  3/13 have no holdup (HL) measurement -- excluded from holdup checks, Um-only
  WC range: 5.20e-05 - 1.23e-03  (PHI_COLLAPSE=0.1 -- every condition sits far below this)
    Um_target        WC  dpdz_measured   psi flow_regime  rho2_val
0   15.128551  0.001226         202.08  0.02          SW      2.57
1   19.724320  0.000726         352.72  0.02          SW      2.67
2   15.103639  0.000903         191.78  0.02          SW      2.57
3   15.130954  0.000724         173.70  0.02          SW      2.57
4   19.890300  0.000518         300.24  0.01          SW      2.66
5   14.937766  0.000520         183.17  0.01          SW      2.56
6   19.866065  0.000305         269.21  0.01          SW      2.65
7   14.974530  0.000303         165.56  0.01          SW      2.55
8   20.263120  0.000154         247.31  0.01          SW      2.52
9   14.852289  0.000154         118.79  0.01          SW      2.43
10  24.961297  0.000052         294.20

### Diagnostic: sweep ALL Fan (2005) conditions for oscillation + collapse (zero-refit Ibarra-fitted closure)

In [7]:
# ── Diagnostic: sweep ALL Fan (2005) conditions for oscillation + collapse ──
# Same methodology as the Ibarra/Russell/MDPI sweeps. Zero-refit: uses the
# Ibarra-fitted w/k1/k2 closure unchanged. Unlike those datasets, rho2 (air
# density) varies per condition here -- read from fan2005_clean's own
# rho2_val column inside the loop, not a fixed global.
#
# Nondimensional boundary conversion (same as generate_real_dataset): D*=1.0
# (L_c=D by construction), p*=p/(rho1*Um^2), d_b*=d_b/D, dt*=dt_seconds*Um/D,
# Re1_ref/Re2_ref/Fr_ref computed per condition from that condition's own
# Um_target -- time_step_learned's viscous-stress term uses 1/Re_ref
# internally now, not mu1/mu2 directly, so these are REQUIRED, not optional
# (their None default raises TypeError -- see CLAUDE.md's nondimensionalization
# note). rho2_star is ALSO computed per-condition here, unlike every other
# dataset, since rho2 itself varies per row.
#
# Expect heavy collapse (see intro cell above for the WC-vs-PHI_COLLAPSE
# numbers) -- this sweep exists to confirm/quantify that, not necessarily to
# find a reachable-conditions set.
#
# REMINDER: D/L/rho1_val/mu1/mu2 must already be set to Fan (2005)'s values
# -- see reset-globals-fan2005-01 above.

import numpy as np
import pickle
import pandas as pd

FAN2005_W, FAN2005_K1, FAN2005_K2 = 1.0, 1.1102, 0.5853   # Ibarra-fitted, zero-refit

N_TOTAL_STEPS_SWEEP       = 370_000   # ~37s -- same full-duration convention as Russell/MDPI
n_chunks_sweep            = 50
chunk_steps_sweep         = N_TOTAL_STEPS_SWEEP // n_chunks_sweep
phi2_range_flag_threshold = 0.3   # coarse placeholder -- see Ibarra/Russell/MDPI sweeps

# Dataset-level dimensionless quantities that DON'T vary per row
D_star    = 1.0
L_star    = L / D
rho1_star = 1.0
d_b_star  = d_b / D

oscillation_sweep_results_fan2005 = []

for _, row in fan2005_clean.iterrows():
    WC_sw, dpdz_sw, Um_sw, psi_sw, rho2_sw = (
        row['WC'], row['dpdz_measured'], row['Um_target'], row['psi'], row['rho2_val']
    )

    print(f"\n{'='*76}")
    print(f"Condition: WC={WC_sw:.6f}  Um={Um_sw:.2f}  dpdz={dpdz_sw:.2f}  "
          f"psi={psi_sw if pd.notna(psi_sw) else 'n/a'}  rho2={rho2_sw:.3f}")
    print(f"{'='*76}")

    p_out_sw = 1.0000e5
    p_in_sw  = p_out_sw + dpdz_sw * L

    # Per-condition reference scales -- Um_sw and rho2_sw are both this
    # condition's own values (rho2 varies per row for this dataset).
    rho2_star  = rho2_sw / rho1_val
    P_c        = rho1_val * Um_sw**2
    p_out_star = p_out_sw / P_c
    p_in_star  = p_in_sw  / P_c
    Re1_ref    = rho1_val * Um_sw * D / mu1
    Re2_ref    = rho1_val * Um_sw * D / mu2
    Fr_ref     = Um_sw / jnp.sqrt(9.81 * D)
    dt_star    = dt_fixed * Um_sw / D

    dx_sw, x_sw = make_grid(L_star, N)
    state_sw = initial_conditions(x_sw, L_star, WC_sw, rho1_star, rho2_star, p_in_star, p_out_star)

    def sweep_scan_step(state, dt):
        new_state = time_step_learned(
            state, dt, dx_sw, drag_coeff, D_star,
            p_in_star, p_out_star, d_b_star, mu1, mu2, theta, WC_sw,
            w=FAN2005_W, k1=FAN2005_K1, k2=FAN2005_K2,
            Re1_ref=Re1_ref, Re2_ref=Re2_ref, Fr_ref=Fr_ref
        )
        return new_state, None

    sweep_scan_jit = jax.jit(lambda s: jax.lax.scan(
        sweep_scan_step, s, jnp.full(chunk_steps_sweep, dt_star)
    )[0])

    t_arr          = np.full(n_chunks_sweep, np.nan)
    phi2_range_arr = np.full(n_chunks_sweep, np.nan)
    slip_range_arr = np.full(n_chunks_sweep, np.nan)
    Um_sim_arr     = np.full(n_chunks_sweep, np.nan)
    phi_min_arr    = np.full(n_chunks_sweep, np.nan)

    i_start_sw, i_end_sw = N // 4, 3 * N // 4

    state_sw_run  = state_sw
    hit_nan_chunk = None

    for c in range(n_chunks_sweep):
        state_sw_run = sweep_scan_jit(state_sw_run)
        state_sw_run['phi1'].block_until_ready()

        has_nan = any(bool(jnp.any(jnp.isnan(v))) for v in state_sw_run.values())
        if has_nan:
            hit_nan_chunk = c
            print(f"  NaN encountered at chunk {c} -- stopping this condition's sweep")
            break

        phi1_c = state_sw_run['phi1']
        phi2_c = state_sw_run['phi2']
        u1_c   = state_sw_run['u1']
        u2_c   = state_sw_run['u2']
        slip_c = u2_c - u1_c

        t_arr[c]          = (c + 1) * chunk_steps_sweep * dt_fixed
        phi2_range_arr[c] = float(jnp.max(phi2_c) - jnp.min(phi2_c))
        slip_range_arr[c] = float(jnp.max(slip_c) - jnp.min(slip_c))
        Um_sim_arr[c]     = float(jnp.mean(
            phi1_c[i_start_sw:i_end_sw] * u1_c[i_start_sw:i_end_sw]
            + phi2_c[i_start_sw:i_end_sw] * u2_c[i_start_sw:i_end_sw]
        ))
        phi_min_arr[c]    = float(jnp.minimum(jnp.min(phi1_c), jnp.min(phi2_c)))

    if np.all(np.isnan(phi2_range_arr)):
        peak_phi2_range = float('nan')
        t_peak          = float('nan')
        oscillating     = False
    else:
        peak_phi2_range = float(np.nanmax(phi2_range_arr))
        t_peak          = float(t_arr[int(np.nanargmax(phi2_range_arr))])
        oscillating     = peak_phi2_range > phi2_range_flag_threshold

    final_Um_sim = float(Um_sim_arr[~np.isnan(Um_sim_arr)][-1]) if np.any(~np.isnan(Um_sim_arr)) else float('nan')

    print(f"  peak phi2_range = {peak_phi2_range:.4f} at t={t_peak:.1f}s   "
          f"flag(>{phi2_range_flag_threshold}) = {oscillating}   "
          f"Um_sim@end = {final_Um_sim:.4f} (target {Um_sw:.2f})"
          + (f"   [NaN at chunk {hit_nan_chunk}]" if hit_nan_chunk is not None else ""))

    oscillation_sweep_results_fan2005.append({
        'WC': WC_sw, 'Um': Um_sw, 'psi': psi_sw, 'rho2_val': rho2_sw,
        't': t_arr, 'phi2_range': phi2_range_arr, 'slip_range': slip_range_arr,
        'Um_sim': Um_sim_arr, 'phi_min': phi_min_arr,
        'peak_phi2_range': peak_phi2_range, 't_peak': t_peak,
        'oscillating_flag': oscillating, 'nan_chunk': hit_nan_chunk,
    })

# ── Summary table ────────────────────────────────────────────────────────────
print(f"\n{'='*76}")
print("Summary across all Fan (2005) conditions")
print(f"{'='*76}")
print(f"{'WC':>10}  {'Um':>5}  {'peak phi2_range':>16}  {'t_peak':>7}  {'flag':>6}  {'nan_chunk':>9}")
for r in oscillation_sweep_results_fan2005:
    print(f"{r['WC']:>10.6f}  {r['Um']:>5.2f}  {r['peak_phi2_range']:>16.4f}  "
          f"{r['t_peak']:>7.1f}  {str(r['oscillating_flag']):>6}  "
          f"{str(r['nan_chunk']) if r['nan_chunk'] is not None else '-':>9}")

# ── Save raw per-chunk time series for reuse by the classification step that would follow ──
with open('fan2005_oscillation_sweep_results.pkl', 'wb') as f:
    pickle.dump(oscillation_sweep_results_fan2005, f)
print("\nSaved per-chunk time series to fan2005_oscillation_sweep_results.pkl")



Condition: WC=0.001226  Um=15.13  dpdz=202.08  psi=0.02  rho2=2.570
  peak phi2_range = 0.0000 at t=0.7s   flag(>0.3) = False   Um_sim@end = 2.0637 (target 15.13)

Condition: WC=0.000726  Um=19.72  dpdz=352.72  psi=0.02  rho2=2.670
  peak phi2_range = 0.0000 at t=0.7s   flag(>0.3) = False   Um_sim@end = 2.1416 (target 19.72)

Condition: WC=0.000903  Um=15.10  dpdz=191.78  psi=0.02  rho2=2.570
  peak phi2_range = 0.0000 at t=0.7s   flag(>0.3) = False   Um_sim@end = 2.0067 (target 15.10)

Condition: WC=0.000724  Um=15.13  dpdz=173.70  psi=0.02  rho2=2.570
  peak phi2_range = 0.0000 at t=0.7s   flag(>0.3) = False   Um_sim@end = 1.8931 (target 15.13)

Condition: WC=0.000518  Um=19.89  dpdz=300.24  psi=0.01  rho2=2.660
  peak phi2_range = 0.0000 at t=0.7s   flag(>0.3) = False   Um_sim@end = 1.9403 (target 19.89)

Condition: WC=0.000520  Um=14.94  dpdz=183.17  psi=0.01  rho2=2.560
  peak phi2_range = 0.0000 at t=0.7s   flag(>0.3) = False   Um_sim@end = 1.9802 (target 14.94)

Condition: WC=0